In [1]:
import glob
import os
import tqdm
import math


import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# import matplotlib.cm as cm
from matplotlib.gridspec import GridSpec
# import mpl_toolkits.axes_grid1
import japanize_matplotlib

import astropy
import astropy.io.fits
import astropy.units as u
import astroquery.vizier
# from astropy.wcs import WCS
from spectral_cube import SpectralCube
import pylab

pylab.rcParams['font.family'] = 'serif'
pylab.rcParams['lines.linewidth'] = 0.5
matplotlib.rcParams["font.family"] = "serif"
matplotlib.rcParams["font.size"] = 15
pylab.rcParams["xtick.direction"] = "in"
pylab.rcParams["ytick.direction"] = "in"

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Hiragino Sans', 'Yu Gothic', 'Meirio', 'Takao', 'IPAexGothic', 'IPAPGothic', 'VL PGothic', 'Noto Sans CJK JP']

In [2]:
import sys
# sys.path.append('/home/elmegreen/galactic_bubble/photoutils/')
from processing import norm_res, normalize_rp, remove_nan, conv, data_view_rectangl, resize
from utils.ssd_model import nm_suppression

# sys.path.append('/home/elmegreen/jupyter/research/Bubble_Analysis/CO_SpitzerBubble/All_Bubble_Analysis/Analysis')
from Function_to_Detect_peak import find_verified_peak, _detect_and_characterize_peak, find_velocity_from_catalog, _check_signal_at_channel, gaussian_filter
from detect_rough_tools import extract_spectral, make_spitzer_fits, load_spitzer_fits, make_momont012_map

In [3]:
viz = astroquery.vizier.Vizier(columns=["*"])
viz.ROW_LIMIT = -1
bub_velocity_table = viz.query_constraints(catalog="J/MNRAS/438/426")[0].to_pandas()

# GLONを0-360度の範囲に正規化
bub_velocity_table['GLON'] = bub_velocity_table['GLON'] % 360
bub_velocity_table['GLON2'] = bub_velocity_table['GLON2'] % 360

print(f"読み込んだバブル速度カタログのエントリ数: {len(bub_velocity_table)}")
print("カタログの列名:", bub_velocity_table.columns.tolist())

読み込んだバブル速度カタログのエントリ数: 818
カタログの列名: ['MWP', 'GLON', 'GLAT', 'Reff', 'GLON2', 'GLAT2', 'Ref', 'VHII', 'D0', 'e_D0', 'r_D0', 'DK', 'e_DK', 'Mark', 'r_Mark', 'Simbad', '_RA.icrs', '_DE.icrs']


In [15]:
all_bubble_catalogue = pd.read_csv('/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/Bubble_Catalogue/infer_catalogue_all.csv')
all_bubble_catalogue['ra_center'] = (all_bubble_catalogue['ra_min'] + all_bubble_catalogue['ra_max'])/2
all_bubble_catalogue['dec_center'] = (all_bubble_catalogue['dec_min'] + all_bubble_catalogue['dec_max'])/2
# all_bubble_catalogue = all_bubble_catalogue[2000:]
all_bubble_catalogue.head()

,Unnamed: 0,dec_min,ra_min,dec_max,ra_max,fits_path,ra_center,dec_center
0,0,-0.746089,309.515787,-0.685465,309.578516,spitzer_30900+0000_rgb,309.547151,-0.715777
1,0,0.532921,308.732721,0.567134,308.766836,spitzer_30900+0000_rgb,308.749778,0.550028
2,0,0.124435,309.003455,0.259821,309.140642,spitzer_30900+0000_rgb,309.072049,0.192128
3,0,-0.490051,307.785009,-0.471448,307.803593,spitzer_30900+0000_rgb,307.794301,-0.480749
4,0,-0.204761,309.235307,-0.176422,309.263029,spitzer_30900+0000_rgb,309.249168,-0.190591


In [16]:
fugin_path_list = sorted(
    glob.glob(
        '/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/Galaxy_Plane/FUGIN/12CO/**'))
print(len(fugin_path_list))
print(fugin_path_list[0])

39
/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/Galaxy_Plane/FUGIN/12CO/FGN_01100+0000_2x2_12CO_v1.00_cube.fits


In [17]:
zeroing_fugin_path_list = sorted(
    glob.glob(
        '/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/Bubble_analysis/All_Bubble_Analysis/Analysis/Fits/Zeroing_Fits/12CO/**'))
print(len(zeroing_fugin_path_list))
print(zeroing_fugin_path_list[0])

39
/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/Bubble_analysis/All_Bubble_Analysis/Analysis/Fits/Zeroing_Fits/12CO/FGN_01100


In [18]:
# 詳細な統計情報を追跡するための変数を初期化
velocity_stats = {
    'catalogue_vel_bubble': 0,
    'catalogue_vel_with_c18o': 0,  # 新規追加：カタログ速度使用かつC18Oピークあり
    'c18o_vel_bubble': 0,
    'co13_vel_bubble': 0,
    'no_vel_bubble': 0,
    'spitzer_error_bubble': 0,
    'total_bubble': 0
}

In [11]:
# 各バブルの詳細情報を記録するリスト
bubble_details = []

for each_path, zeroing_each_path in tqdm.tqdm(zip(fugin_path_list, zeroing_fugin_path_list)):
    # パスやディレクトリの設定
    base_dir = os.path.dirname(os.path.dirname(each_path))
    region_name = each_path.split('/')[-1].split('+')[0]
    output_fig_dir = os.path.join('Spectral_fig', region_name)
    os.makedirs(output_fig_dir, exist_ok=True)

    Zeroing_12CO_fits_path = glob.glob(
        '/'.join(zeroing_each_path.split('/')[:-2]) + "/12CO/" + region_name + f"/{region_name}**.fits")[0]
    Zeroing_13CO_fits_path = glob.glob(
        '/'.join(zeroing_each_path.split('/')[:-2]) + "/13CO/" + region_name + f"/{region_name}**.fits")[0]
    Zeroing_C18O_fits_path = glob.glob(
        '/'.join(zeroing_each_path.split('/')[:-2]) + "/C18O/" + region_name + f"/{region_name}**.fits")[0]
    Zeroing_fugin_cube_fits_12CO = astropy.io.fits.open(Zeroing_12CO_fits_path)[0]
    Zeroing_fugin_cube_fits_13CO = astropy.io.fits.open(Zeroing_13CO_fits_path)[0]
    Zeroing_fugin_cube_fits_C18O = astropy.io.fits.open(Zeroing_C18O_fits_path)[0]

    # FITSファイルのパスを構築
    _12CO_fits_path = os.path.join(base_dir, "12CO", f"{region_name}+0000_2x2_12CO_v1.00_cube.fits")
    _13CO_fits_path = os.path.join(base_dir, "13CO", f"{region_name}+0000_2x2_13CO_v1.00_cube.fits")
    C18O_fits_path = glob.glob(os.path.join(base_dir, "C18O", f"{region_name}+0000_2x2_C18O_v1.*.fits"))[0]

    # FITSファイルを開く
    fugin_cube_fits_12CO = astropy.io.fits.open(_12CO_fits_path)[0]
    fugin_cube_fits_13CO = astropy.io.fits.open(_13CO_fits_path)[0]
    fugin_cube_fits_C18O = astropy.io.fits.open(C18O_fits_path)[0]
    
    # WCSと速度軸の情報を抽出
    w_co = astropy.wcs.WCS(fugin_cube_fits_12CO.header)
    cube = SpectralCube.read(fugin_cube_fits_12CO)
    vaxis = cube.spectral_axis.to_value(u.km/u.s)
    dv = abs(fugin_cube_fits_12CO.header['CDELT3']) / 1000.0

    # カタログフィルタリング
    ny, nx = fugin_cube_fits_12CO.data.shape[1:3]
    glon_min, glat_min, _ = w_co.all_pix2world(nx, 0, 0, 0)
    glon_max, glat_max, _ = w_co.all_pix2world(0, ny, 0, 0)
    all_bubble_catalogue_selected = all_bubble_catalogue.query(
        f"{glon_min} <= ra_center <= {glon_max} and {glat_min} <= dec_center <= {glat_max}"
    ).reset_index()

    # --- レイアウト設定 ---
    bubbles_per_row = 2
    rows_per_page = 100  # 1ページに表示する行数
    bubbles_per_page = bubbles_per_row * rows_per_page
    total_bubbles = len(all_bubble_catalogue_selected)
    num_pages = math.ceil(total_bubbles / bubbles_per_page)
        
    # キャッシュ初期化
    spitzer_path_cached = None
    
    # ページごとのループ
    for page in range(num_pages):
        start_idx = page * bubbles_per_page
        end_idx = min(start_idx + bubbles_per_page, total_bubbles)
        current_page_count = end_idx - start_idx
        
        # ページ内の必要行数を計算
        current_rows_in_page = math.ceil(current_page_count / bubbles_per_row)
        
        # ページ全体のFigure作成
        fig = plt.figure(figsize=(15*bubbles_per_row, 8 * current_rows_in_page))
        
        # サブフィギュアの作成
        subfigs_obj = fig.subfigures(current_rows_in_page, bubbles_per_row, wspace=0.1, hspace=0.1)
        
        # 【修正ポイント】戻り値が配列なら平坦化し、単一オブジェクトならリストに包む
        if isinstance(subfigs_obj, np.ndarray):
            subfigs = subfigs_obj.flatten()
        else:
            subfigs = [subfigs_obj]
    
        for p_idx, subfig in enumerate(subfigs):
            i_in_catalogue = start_idx + p_idx
            if i_in_catalogue >= total_bubbles:
                subfig.set_visible(False)
                continue

            each_catalogue = all_bubble_catalogue_selected.iloc[i_in_catalogue]
            i = each_catalogue.name # 元のインデックス
            
            # --- データ処理ロジック (省略なし) ---
            velocity_stats['total_bubble'] += 1
    
            bubble_info = {
                'region': region_name,
                'catalogue_index': i,
                'ra_center': each_catalogue['ra_center'],
                'dec_center': each_catalogue['dec_center'],
                'velocity_source': None,
                'v_peak': None,
                'fwhm_vel': None,
                'has_c18o_peak': False,
                'status': None
            }
    
            # Spitzerデータのロード（キャッシュを利用）
            current_spitzer_path = os.path.join("/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/Galaxy_Plane/spitzer_data", str(each_catalogue['fits_path'])+'/')
            if current_spitzer_path != spitzer_path_cached:
                spitzer_rfits, spitzer_gfits, spitzer_data, w_spitzer = load_spitzer_fits(current_spitzer_path)
                spitzer_path_cached = current_spitzer_path
    
            # スペクトルデータを抽出
            cut_data_12CO = extract_spectral(w_co, fugin_cube_fits_12CO, each_catalogue)
            cut_data_13CO = extract_spectral(w_co, fugin_cube_fits_13CO, each_catalogue)
            cut_data_C18O = extract_spectral(w_co, fugin_cube_fits_C18O, each_catalogue)
    
            zeroing_cut_data_12CO = extract_spectral(w_co, Zeroing_fugin_cube_fits_12CO, each_catalogue)
            zeroing_cut_data_13CO = extract_spectral(w_co, Zeroing_fugin_cube_fits_13CO, each_catalogue)
            zeroing_cut_data_C18O = extract_spectral(w_co, Zeroing_fugin_cube_fits_C18O, each_catalogue)
    
            # 平均スペクトルを計算
            mean_data_12CO = np.nanmean(cut_data_12CO, axis=(1, 2))
            mean_data_13CO = np.nanmean(cut_data_13CO, axis=(1, 2))
            mean_data_C18O = np.nanmean(cut_data_C18O, axis=(1, 2))
    
            # C18Oのピーク検証
            c18o_v_peak, c18o_t_peak, c18o_fwhm_vel, c18o_peak_channel = find_verified_peak(
                c18o_spec=mean_data_C18O,
                co12_spec=mean_data_12CO,
                co13_spec=mean_data_13CO,
                vaxis=vaxis, dv=dv
            )
    
            bubble_info['has_c18o_peak'] = c18o_v_peak is not None
    
            # [cite_start]カタログ（Beaumont & Williams 2014等）から速度情報を検索
            catalog_v_peak, catalog_fwhm_vel, catalog_info = find_velocity_from_catalog(
                each_catalogue['ra_center'], 
                each_catalogue['dec_center'], 
                bub_velocity_table,
                search_radius=(each_catalogue['dec_max'] - each_catalogue['dec_min'])/2
            )
    
            if catalog_v_peak is not None:
                used_tracer = f"Catalog (Row {catalog_info['catalog_index']})"
                v_peak = catalog_v_peak
                fwhm_vel = catalog_fwhm_vel
                t_peak = np.nan
                peak_channel = np.argmin(np.abs(vaxis - v_peak))
                
                if bubble_info['has_c18o_peak'] and abs(c18o_v_peak - catalog_v_peak) <= 10.0:
                    velocity_stats['catalogue_vel_with_c18o'] += 1
                else:
                    bubble_info['has_c18o_peak'] = False 
                
                velocity_stats['catalogue_vel_bubble'] += 1
                bubble_info.update({'velocity_source': 'Catalog', 'v_peak': v_peak, 'fwhm_vel': fwhm_vel, 'status': 'Success'})
                
            elif c18o_v_peak is not None:
                used_tracer = "C18O (verified)"
                v_peak, t_peak, fwhm_vel, peak_channel = c18o_v_peak, c18o_t_peak, c18o_fwhm_vel, c18o_peak_channel
                velocity_stats['c18o_vel_bubble'] += 1
                bubble_info.update({'velocity_source': 'C18O', 'v_peak': v_peak, 'fwhm_vel': fwhm_vel, 'status': 'Success'})
                
            else:
                used_tracer = "13CO"
                v_peak, t_peak, fwhm_vel, peak_channel = _detect_and_characterize_peak(
                    mean_data_13CO, vaxis, dv, height_factor=5.0, prominence_factor=3.0
                )
                if v_peak is not None:
                    velocity_stats['co13_vel_bubble'] += 1
                    bubble_info.update({'velocity_source': '13CO', 'v_peak': v_peak, 'fwhm_vel': fwhm_vel, 'status': 'Success'})
                else:
                    velocity_stats['no_vel_bubble'] += 1
                    bubble_info.update({'velocity_source': 'None', 'status': 'No velocity detected'})
                    subfig.text(0.5, 0.5, f"No velocity detected\nIdx: {i}", ha='center', va='center', fontsize=12)
                    bubble_details.append(bubble_info)
                    continue
    
            # 速度範囲の設定
            range_start_vel = v_peak - fwhm_vel * 2.5
            range_end_vel = v_peak + fwhm_vel * 2.5
            range_indices = np.where((vaxis >= range_start_vel) & (vaxis <= range_end_vel))[0]
    
            # Momentマップ作成 (FUGINデータを使用)
            datadict_12CO = make_momont012_map(fugin_cube_fits_12CO, w_co, zeroing_cut_data_12CO[range_indices], vaxis[range_indices], v_center=v_peak)
            datadict_13CO = make_momont012_map(fugin_cube_fits_13CO, astropy.wcs.WCS(fugin_cube_fits_13CO.header), zeroing_cut_data_13CO[range_indices], vaxis[range_indices], v_center=v_peak)
            datadict_C18O = make_momont012_map(fugin_cube_fits_C18O, astropy.wcs.WCS(fugin_cube_fits_C18O.header), zeroing_cut_data_C18O[range_indices], vaxis[range_indices], v_center=v_peak)
    
            # Spitzer画像の準備
            new_hdu_list_r, new_hdu_list_g = make_spitzer_fits(spitzer_rfits, spitzer_gfits, w_spitzer, spitzer_data, each_catalogue)
            if new_hdu_list_r == 0:
                velocity_stats['spitzer_error_bubble'] += 1
                bubble_info['status'] = 'Spitzer error'
                subfig.text(0.5, 0.5, f"Spitzer Error\nIdx: {i}", ha='center', va='center', fontsize=12)
                bubble_details.append(bubble_info)
                continue
                
            cut_spitzer = np.concatenate([
                new_hdu_list_r[0].data[:,:,None],
                new_hdu_list_g[0].data[:,:,None],
                np.zeros(new_hdu_list_r[0].data.shape)[:,:,None]
            ], axis=2)
    
            # --- プロット処理 ---
            v_peak_str = f"{v_peak:.1f}" if v_peak is not None else "N/A"
            c18o_v_peak_str = f"{c18o_v_peak:.1f}" if c18o_v_peak is not None else "N/A"
    
            if "Catalog" in used_tracer:
                title = (f"Peak from Hou et al. 2013, "
                         f"V_HII={v_peak_str} km/s, C18O peak={c18o_v_peak_str} km/s")
            else:
                title = (f"Catalogue: {i}, Bubble: {each_catalogue.get('name', 'N/A')}, "
                         f"Peak from {used_tracer} at V_lsr={v_peak_str} km/s")

            # subfigにタイトルを設定
            subfig.suptitle(title, fontsize=20, fontweight='bold', y=0.98)
            
            gs = GridSpec(3, 7, figure=subfig, width_ratios=[1, 1, 1, 1, 1, 1, 2], 
                          wspace=0.8, hspace=0.3, left=0.05, right=0.95, top=0.90, bottom=0.08)
    
            # 1. Spitzer赤外線画像
            ax = subfig.add_subplot(gs[:3, :3], projection=astropy.wcs.WCS(new_hdu_list_r[0].header))
            ax.imshow(cut_spitzer)
            ax.tick_params(labelsize=8)
            ax.set_xlabel('Galactic Longitude', fontsize=15)
            ax.set_ylabel('Galactic Latitude', fontsize=15)
    
            # 2. スペクトル (12CO, 13CO, C18O)
            spectral_data_list = [("12CO", mean_data_12CO), ("13CO", mean_data_13CO), ("C18O", mean_data_C18O)]
            for idx, (label, spec_data) in enumerate(spectral_data_list):
                ax = subfig.add_subplot(gs[idx, 3:6])
                ax.plot(vaxis, spec_data, "k", label=label, drawstyle='steps-mid')
                if len(range_indices) > 0:
                    ax.plot(vaxis[range_indices], spec_data[range_indices], "o", color='red', markersize=2)
                    ax.axvspan(range_start_vel, range_end_vel, alpha=0.2, color='red')
        
                ax.axvline(v_peak, color='blue', ls='--', lw=1, label=f'Peak: {v_peak_str} km/s')
                if bubble_info['has_c18o_peak'] and idx == 2:
                    ax.axvline(c18o_v_peak, color='green', ls=':', lw=2, label=f'C18O peak: {c18o_v_peak_str} km/s')
        
                ax.set_xlim([-80, 80])
                ax.tick_params(axis='both', labelsize=12)
                ax.set_title(label, fontsize=15)
                ax.set_ylabel("Mean Tmb [K]", fontsize=12)
                ax.legend(fontsize=10, loc='upper right')
    
            # 3. 積分強度マップ (Moment 0)
            moms = [datadict_12CO, datadict_13CO, datadict_C18O]
            for idx, mdata in enumerate(moms):
                ax = subfig.add_subplot(gs[idx, 6:7])
                mdata = mdata['moment0']
                ax.imshow(mdata, origin='lower', interpolation='none', cmap='viridis')
    
                # コントア作成
                x_width = mdata.shape[1]
                y_width = mdata.shape[0]
                x = np.linspace(0, x_width, x_width)
                y = np.linspace(0, y_width, y_width)
                X, Y = np.meshgrid(x, y)
                sig1 = 1 / (2 * (np.log(2)) ** 0.5)
    
                # 修正：resize(conv(...)) の結果が正しく渡るように
                try:
                    spitzer_contour_data = resize(conv(int(x_width), sig1, cut_spitzer), (int(y_width), int(x_width)))[:,:,1]
                    contour = ax.contour(X, Y, spitzer_contour_data, levels=[0.1, 0.3, 0.5], colors=['w'], linewidths=3)
                    ax.clabel(contour, inline=True, fontsize=12)
                except Exception as e:
                    print(f"Contour error at index {i}: {e}")
    
                r_pix = x_width/4
                center_pix = x_width/2
                tick_pos = center_pix + np.array([-1, 0, 1]) * r_pix
                ax.set_xticks(tick_pos); ax.set_xticklabels(['-R', '0', 'R'])
                ax.set_yticks(tick_pos); ax.set_yticklabels(['-R', '0', 'R'])
                ax.set_title(['12CO', '13CO', 'C18O'][idx] + ' Moment 0', fontsize=15)
    
            bubble_details.append(bubble_info)
    
        # ページ保存
        # dir_name = f"FUGIN_Bubble_Profile/{region_name}"
        # os.makedirs(dir_name, exist_ok=True)
    
        out_name = f"FUGIN_Bubble_Profile/Summary_{region_name}.png"
        
        # plt.savefig(out_name, dpi=72, bbox_inches='tight')
        plt.close(fig)
        print(f"Generated: {out_name}")

0it [00:00, ?it/s]

Contour error at index 6: division by zero
Contour error at index 6: division by zero
Contour error at index 6: division by zero


1it [00:17, 17.07s/it]

Generated: FUGIN_Bubble_Profile/Summary_FGN_01100.png


2it [00:27, 12.88s/it]

...

15it [02:15,  9.05s/it]

KeyboardInterrupt



Error in callback <function _draw_all_if_interactive at 0x747aedbd1630> (for post_execute), with arguments args (),kwargs {}:



KeyboardInterrupt



Error in callback <function flush_figures at 0x74797bc1f250> (for post_execute), with arguments args (),kwargs {}:


KeyboardInterrupt: 

In [22]:
import os
import math
import glob
import numpy as np
import tqdm
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import astropy.wcs
import astropy.io.fits
import astropy.units as u
from spectral_cube import SpectralCube
from skimage.transform import resize

# 統計情報を初期化
velocity_stats = {
    'total_bubble': 0,
    'catalogue_vel_bubble': 0,
    'catalogue_vel_with_c18o': 0,
    'c18o_vel_bubble': 0,
    'co13_vel_bubble': 0,
    'no_vel_bubble': 0,
    'spitzer_error_bubble': 0
}

# 各バブルの詳細情報を記録するリスト
bubble_details = []

# --- レイアウト設定 ---
bubbles_per_row = 2

# パスリストの存在確認（環境に合わせて調整してください）
# fugin_path_list, zeroing_fugin_path_list, all_bubble_catalogue, bub_velocity_table, vaxis, dv などの変数は定義済みと想定

print(f"Starting analysis for {len(fugin_path_list)} regions...")

for each_path, zeroing_each_path in tqdm.tqdm(zip(fugin_path_list, zeroing_fugin_path_list), total=len(fugin_path_list)):
    # パスやディレクトリの設定
    base_dir = os.path.dirname(os.path.dirname(each_path))
    region_name = each_path.split('/')[-1].split('+')[0]
    
    # 処理開始のデバッグプリント
    print(f"\n[Processing Region: {region_name}]")

    # FITSファイルのパスを構築（各領域で必要なファイルを検索）
    try:
        z_base = '/'.join(zeroing_each_path.split('/')[:-2])
        Zeroing_12CO_fits_path = glob.glob(f"{z_base}/12CO/{region_name}/{region_name}**.fits")[0]
        Zeroing_13CO_fits_path = glob.glob(f"{z_base}/13CO/{region_name}/{region_name}**.fits")[0]
        Zeroing_C18O_fits_path = glob.glob(f"{z_base}/C18O/{region_name}/{region_name}**.fits")[0]
        
        Zeroing_fugin_cube_fits_12CO = astropy.io.fits.open(Zeroing_12CO_fits_path)[0]
        Zeroing_fugin_cube_fits_13CO = astropy.io.fits.open(Zeroing_13CO_fits_path)[0]
        Zeroing_fugin_cube_fits_C18O = astropy.io.fits.open(Zeroing_C18O_fits_path)[0]

        _12CO_fits_path = os.path.join(base_dir, "12CO", f"{region_name}+0000_2x2_12CO_v1.00_cube.fits")
        _13CO_fits_path = os.path.join(base_dir, "13CO", f"{region_name}+0000_2x2_13CO_v1.00_cube.fits")
        C18O_fits_path = glob.glob(os.path.join(base_dir, "C18O", f"{region_name}+0000_2x2_C18O_v1.*.fits"))[0]

        fugin_cube_fits_12CO = astropy.io.fits.open(_12CO_fits_path)[0]
        fugin_cube_fits_13CO = astropy.io.fits.open(_13CO_fits_path)[0]
        fugin_cube_fits_C18O = astropy.io.fits.open(C18O_fits_path)[0]
    except Exception as e:
        print(f"  --> SKIP: FITS file acquisition failed for {region_name}. Error: {e}")
        continue

    # WCSと速度軸の情報を抽出
    w_co = astropy.wcs.WCS(fugin_cube_fits_12CO.header)
    cube = SpectralCube.read(fugin_cube_fits_12CO)
    vaxis = cube.spectral_axis.to_value(u.km/u.s)
    dv = abs(fugin_cube_fits_12CO.header['CDELT3']) / 1000.0

    # カタログフィルタリング（FITSの座標範囲を正確に取得）
    ny, nx = fugin_cube_fits_12CO.data.shape[1:3]
    # ピクセル角（0,0）から（nx,ny）のワールド座標を取得
    c_min = w_co.all_pix2world(nx, 0, 0, 0)
    c_max = w_co.all_pix2world(0, ny, 0, 0)
    
    glon_range = [min(c_min[0], c_max[0]), max(c_min[0], c_max[0])]
    glat_range = [min(c_min[1], c_max[1]), max(c_min[1], c_max[1])]
    
    print(f"  FITS Bounds: GLON [{glon_range[0]:.3f}, {glon_range[1]:.3f}], GLAT [{glat_range[0]:.3f}, {glat_range[1]:.3f}]")

    # カタログから範囲内を抽出
    # カタログの列名が ra_center, dec_center であると想定（中身が銀経・銀緯であることを前提）
    all_bubble_catalogue_selected = all_bubble_catalogue.query(
        f"{glon_range[0]} <= ra_center <= {glon_range[1]} and {glat_range[0]} <= dec_center <= {glat_range[1]}"
    ).reset_index(drop=True)

    total_bubbles_in_region = len(all_bubble_catalogue_selected)
    print(f"  Result: Found {total_bubbles_in_region} bubbles in this FITS region.")

    if total_bubbles_in_region == 0:
        print(f"  --> SKIP: No bubbles in {region_name} based on coordinate filter.")
        continue

    # 必要な行数を計算
    num_rows = math.ceil(total_bubbles_in_region / bubbles_per_row)
    spitzer_path_cached = None

    # Figureの作成（全バブル分を1枚に）
    fig = plt.figure(figsize=(15 * bubbles_per_row, 8 * num_rows))
    
    # サブフィギュアの作成
    subfigs_obj = fig.subfigures(num_rows, bubbles_per_row, wspace=0.1, hspace=0.1)
    
    if isinstance(subfigs_obj, np.ndarray):
        subfigs = subfigs_obj.flatten()
    else:
        subfigs = [subfigs_obj]

    # バブルごとの処理ループ
    for p_idx, subfig in enumerate(subfigs):
        if p_idx >= total_bubbles_in_region:
            subfig.set_visible(False)
            continue

        each_catalogue = all_bubble_catalogue_selected.iloc[p_idx]
        i = each_catalogue.name 

        # --- データ処理ロジック ---
        velocity_stats['total_bubble'] += 1
        bubble_info = {
            'region': region_name,
            'catalogue_index': i,
            'ra_center': each_catalogue['ra_center'],
            'dec_center': each_catalogue['dec_center'],
            'velocity_source': None,
            'v_peak': None,
            'fwhm_vel': None,
            'has_c18o_peak': False,
            'status': None
        }

        # Spitzerデータのロード（キャッシュを利用）
        current_spitzer_path = os.path.join("/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/Galaxy_Plane/spitzer_data", str(each_catalogue['fits_path'])+'/')
        if current_spitzer_path != spitzer_path_cached:
            try:
                spitzer_rfits, spitzer_gfits, spitzer_data, w_spitzer = load_spitzer_fits(current_spitzer_path)
                spitzer_path_cached = current_spitzer_path
            except Exception as e:
                subfig.text(0.5, 0.5, f"Spitzer Error\n{e}", ha='center', va='center')
                continue

        # スペクトル抽出
        cut_data_12CO = extract_spectral(w_co, fugin_cube_fits_12CO, each_catalogue)
        cut_data_13CO = extract_spectral(w_co, fugin_cube_fits_13CO, each_catalogue)
        cut_data_C18O = extract_spectral(w_co, fugin_cube_fits_C18O, each_catalogue)
        zeroing_cut_data_12CO = extract_spectral(w_co, Zeroing_fugin_cube_fits_12CO, each_catalogue)
        zeroing_cut_data_13CO = extract_spectral(w_co, Zeroing_fugin_cube_fits_13CO, each_catalogue)
        zeroing_cut_data_C18O = extract_spectral(w_co, Zeroing_fugin_cube_fits_C18O, each_catalogue)

        mean_data_12CO = np.nanmean(cut_data_12CO, axis=(1, 2))
        mean_data_13CO = np.nanmean(cut_data_13CO, axis=(1, 2))
        mean_data_C18O = np.nanmean(cut_data_C18O, axis=(1, 2))

        # 速度ピーク判定
        c18o_v_peak, c18o_t_peak, c18o_fwhm_vel, _ = find_verified_peak(mean_data_C18O, mean_data_12CO, mean_data_13CO, vaxis, dv)
        bubble_info['has_c18o_peak'] = c18o_v_peak is not None

        catalog_v_peak, catalog_fwhm_vel, catalog_info = find_velocity_from_catalog(
            each_catalogue['ra_center'], each_catalogue['dec_center'], bub_velocity_table,
            search_radius=(each_catalogue['dec_max'] - each_catalogue['dec_min'])/2
        )

        if catalog_v_peak is not None:
            used_tracer, v_peak, fwhm_vel = "Catalog", catalog_v_peak, catalog_fwhm_vel
            if not (bubble_info['has_c18o_peak'] and abs(c18o_v_peak - catalog_v_peak) <= 10.0):
                bubble_info['has_c18o_peak'] = False
            velocity_stats['catalogue_vel_bubble'] += 1
        elif c18o_v_peak is not None:
            used_tracer, v_peak, fwhm_vel = "C18O", c18o_v_peak, c18o_fwhm_vel
            velocity_stats['c18o_vel_bubble'] += 1
        else:
            used_tracer = "13CO"
            v_peak, _, fwhm_vel, _ = _detect_and_characterize_peak(mean_data_13CO, vaxis, dv)
            if v_peak is None:
                velocity_stats['no_vel_bubble'] += 1
                subfig.text(0.5, 0.5, f"No Velocity Detected\nIdx: {p_idx}", ha='center', va='center', fontsize=20)
                continue
            velocity_stats['co13_vel_bubble'] += 1

        bubble_info.update({'velocity_source': used_tracer, 'v_peak': v_peak, 'fwhm_vel': fwhm_vel, 'status': 'Success'})

        # Momentマップ & Spitzer準備
        range_idx = np.where((vaxis >= v_peak - fwhm_vel*2.5) & (vaxis <= v_peak + fwhm_vel*2.5))[0]
        d12 = make_momont012_map(fugin_cube_fits_12CO, w_co, zeroing_cut_data_12CO[range_idx], vaxis[range_idx], v_center=v_peak)
        d13 = make_momont012_map(fugin_cube_fits_13CO, w_co, zeroing_cut_data_13CO[range_idx], vaxis[range_idx], v_center=v_peak)
        d18 = make_momont012_map(fugin_cube_fits_C18O, w_co, zeroing_cut_data_C18O[range_idx], vaxis[range_idx], v_center=v_peak)

        new_hr, new_hg = make_spitzer_fits(spitzer_rfits, spitzer_gfits, w_spitzer, spitzer_data, each_catalogue)
        if new_hr == 0:
            subfig.text(0.5, 0.5, "Spitzer Image Error", ha='center', va='center')
            continue
        cut_spitzer = np.concatenate([new_hr[0].data[:,:,None], new_hg[0].data[:,:,None], np.zeros(new_hr[0].data.shape)[:,:,None]], axis=2)

        # --- プロット描画 ---
        v_peak_str = f"{v_peak:.1f}"
        subfig.suptitle(f"Region: {region_name}, Idx: {p_idx}, Peak: {used_tracer} ({v_peak_str} km/s)", fontsize=22, fontweight='bold', y=0.96)

        gs = GridSpec(3, 7, figure=subfig, width_ratios=[1, 1, 1, 1, 1, 1, 2], wspace=0.8, hspace=0.3, left=0.05, right=0.95, top=0.90, bottom=0.08)

        # 1. Spitzer画像
        ax_sp = subfig.add_subplot(gs[:3, :3], projection=astropy.wcs.WCS(new_hr[0].header))
        ax_sp.imshow(cut_spitzer)
        ax_sp.set_xlabel('GLON', fontsize=15); ax_sp.set_ylabel('GLAT', fontsize=15)

        # 2. スペクトル
        for idx, (label, s_data) in enumerate([("12CO", mean_data_12CO), ("13CO", mean_data_13CO), ("C18O", mean_data_C18O)]):
            ax_s = subfig.add_subplot(gs[idx, 3:6])
            ax_s.plot(vaxis, s_data, "k", drawstyle='steps-mid')
            ax_s.axvline(v_peak, color='blue', ls='--', lw=1)
            ax_s.axvspan(vaxis[range_idx[0]], vaxis[range_idx[-1]], alpha=0.2, color='red')
            ax_s.set_xlim([-100, 100])
            ax_s.set_title(label, fontsize=15)

        # 3. Moment 0
        for idx, m_dict in enumerate([d12, d13, d18]):
            ax_m = subfig.add_subplot(gs[idx, 6:7])
            m_data = m_dict['moment0']
            ax_m.imshow(m_data, origin='lower', cmap='viridis')
            
            # コンター処理
            try:
                x_w, y_w = m_data.shape[1], m_data.shape[0]
                X, Y = np.meshgrid(np.linspace(0, x_w, x_w), np.linspace(0, y_w, y_w))
                sig1 = 1 / (2 * (np.log(2)) ** 0.5)
                sp_contour = resize(conv(int(x_w), sig1, cut_spitzer), (int(y_w), int(x_w)))[:,:,1]
                ax_m.contour(X, Y, sp_contour, levels=[0.1, 0.3, 0.5], colors=['w'], linewidths=2)
            except: pass

            ax_m.set_xticks([]); ax_m.set_yticks([])
            ax_m.set_title(['12CO', '13CO', 'C18O'][idx] + ' Mom0', fontsize=15)

        bubble_details.append(bubble_info)

    # 保存
    out_name = f"FUGIN_Bubble_Profile/Summary_{region_name}.png"
    plt.savefig(out_name, dpi=72, bbox_inches='tight')
    plt.close(fig)
    print(f"  [SUCCESS] Saved {region_name} with {total_bubbles_in_region} bubbles.")

print("\nAll processing complete.")

Starting analysis for 39 regions...


  0%|                                                                                   | 0/39 [00:00<?, ?it/s]


[Processing Region: FGN_01100]
  FITS Bounds: GLON [9.998, 12.000], GLAT [-1.000, 1.002]
  Result: Found 64 bubbles in this FITS region.


  3%|█▉                                                                         | 1/39 [00:24<15:39, 24.73s/it]

  [SUCCESS] Saved FGN_01100 with 64 bubbles.

[Processing Region: FGN_01200]
  FITS Bounds: GLON [10.998, 13.000], GLAT [-1.000, 1.002]
  Result: Found 65 bubbles in this FITS region.


...

  [SUCCESS] Saved FGN_04700 with 21 bubbles.

[Processing Region: FGN_04800]
  FITS Bounds: GLON [46.998, 49.000], GLAT [-1.000, 1.002]
  Result: Found 34 bubbles in this FITS region.


 97%|████████████████████████████████████████████████████████████████████████  | 38/39 [11:57<00:15, 15.68s/it]

  [SUCCESS] Saved FGN_04800 with 34 bubbles.

[Processing Region: FGN_04900]
  FITS Bounds: GLON [47.998, 50.000], GLAT [-1.000, 1.002]
  Result: Found 83 bubbles in this FITS region.


100%|██████████████████████████████████████████████████████████████████████████| 39/39 [12:32<00:00, 19.29s/it]

  [SUCCESS] Saved FGN_04900 with 83 bubbles.

All processing complete.
